## IDS Modeling

Network Duplicate, we need no trainings here

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from phis_init import phi_init

from Channel_Current_Ich import (
    channel_integral_I_phi,
    channel_current_Ich
)

class DeltaPhisPINN(nn.Module):

    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(2, 8),
            nn.Tanh(),
            nn.Linear(8, 1)
        )

    def forward(self, Vgs, Vds):

        Vgs = Vgs.reshape(-1, 1)
        Vds = Vds.reshape(-1, 1)

        x = torch.cat(
            [Vgs, Vds],
            dim=1
        )

        return self.net(x)


device = (
    "xpu"
    if torch.xpu.is_available()
    else "cpu"
)

dtype = torch.float32

print("device =", device)
print("dtype  =", dtype)

## Parameters transfer
Need to be correspondent

In [ ]:
# ============================================================
# Physical parameters
# ============================================================

T = 300.0                         # K

NA = 1e16                        # cm^-3

# Absolute permittivity of 4H-SiC
eps_sic = 9.26 * 8.854e-14       # F/cm

Cox = 7e-8                       # F/cm^2

Vfbs0 = -1.0                     # V

Dit_mid = 2e11                   # cm^-2 eV^-1

Dit_edge = 2e13                  # cm^-2 eV^-1

sigma_it = 0.1                   # eV

Eg = 3.26                        # eV

Ec_minus_Ei = Eg / 2.0           # eV

Qox = 1.602e-7                   # C/cm^2


# ============================================================
# Valid voltage range of trained PINN
# ============================================================

VGS_MIN = -10.0
VGS_MAX = 30.0

VDS_MIN = 0.0
VDS_MAX = 15.0

In [ ]:
Model = DeltaPhisPINN().to(
    device=device,
    dtype=dtype
)

Model.load_state_dict(
    torch.load(
        "phis_PINN.pth",
        map_location=device
    )
)

Model.eval()

print("Surface-potential PINN loaded.")



Redefine Outputs

In [ ]:
def predict_phis(
    Model,
    Vgs,
    Vds
):

    # --------------------------------------------------------
    # Ensure tensor shape
    # --------------------------------------------------------

    Vgs = Vgs.reshape(-1, 1)
    Vds = Vds.reshape(-1, 1)


    # --------------------------------------------------------
    # Drain-side quasi-Fermi potential
    # --------------------------------------------------------

    phi_f = Vds


    # --------------------------------------------------------
    # Analytical initial surface potential
    # --------------------------------------------------------

    phis_init = phi_init(
        Vgs,
        phi_f,
        T,
        NA,
        eps_sic,
        Cox,
        Vfbs0,
        Dit_mid,
        Dit_edge,
        sigma_it,
        Eg,
    ).reshape(-1, 1)


    # --------------------------------------------------------
    # PINN correction
    # --------------------------------------------------------

    delta_phis = Model(
        Vgs,
        Vds
    )


    # --------------------------------------------------------
    # Final surface potential
    # --------------------------------------------------------

    phis = (
        phis_init
        + delta_phis
    )


    return phis

In [ ]:
def predict_phis_source(
    Model,
    Vgs
):

    Vds_source = torch.zeros_like(
        Vgs
    )

    phis_s0 = predict_phis(
        Model,
        Vgs,
        Vds_source
    )

    return phis_s0

def predict_phis_drain(
    Model,
    Vgs,
    Vds
):

    phis_sL = predict_phis(
        Model,
        Vgs,
        Vds
    )

    return phis_sL

In [ ]:
VGS_TEST = 15.0
VDS_TEST = 5.0

Vgs_test = torch.tensor(
    [[VGS_TEST]],
    dtype=dtype,
    device=device
)

Vds_test = torch.tensor(
    [[VDS_TEST]],
    dtype=dtype,
    device=device
)


with torch.no_grad():

    phis_s0_test = predict_phis_source(
        Model,
        Vgs_test
    )

    phis_sL_test = predict_phis_drain(
        Model,
        Vgs_test,
        Vds_test
    )


print(
    "Vgs      =",
    Vgs_test.item(),
    "V"
)

print(
    "Vds      =",
    Vds_test.item(),
    "V"
)

print(
    "phis_s0  =",
    phis_s0_test.item(),
    "V"
)

print(
    "phis_sL  =",
    phis_sL_test.item(),
    "V"
)

## Params Definition


In [ ]:
# First-stage IV parameters

W_eff_cm = 1.0

Lch_cm = 1.0e-4

mu_eff_cm2_Vs = 20.0

lambda_clm = 0.0